# T7 · Reading the curve

**Tiny Transformer, Unit 2, module T7.** Every 500 steps, `train.py` checked the model on both texts and wrote the
scores down: 201 checks, its log. Here you read that log as a curve: how fast it falls, its ups and downs, the gap between
the two texts' curves; then you have best.pt write, and count its words found in the training text.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/T7-training-progress.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook T7-training-progress.ipynb`.

**Computed here:** the share of the fall at any step, the checks worse than the one before, the gap at every check, and best.pt's
writing with its words looked up. **Checked against the run:** the file it saved, its log, and the course's numbers, which the
notebook asserts.

Laid out like the page: two sections, four chapters, and under each chapter its page's three steps (Picture it, Watch it in our
run, Read and run the code), then the code to run and a "your turn".

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import hashlib, json, os, urllib.request
import torch

torch.set_num_threads(4)     # as the run did
print(torch.__version__)

2.10.0+cpu


**The text, the model and this module's code.** Tiny Shakespeare, pinned by its sha256; then the model's file, `model.py`, copied
in whole; then Unit 2's teaching functions, `code/training_steps.py`, copied in whole: T7's are the last five.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
#@title Unit 2's teaching functions: code/training_steps.py, copied whole (run me) { display-mode: "form" }
"""Unit 2 · Inside the learning loop: the recorded run's training, station by station, done explicitly.

The teaching functions of Unit 2 (each T notebook carries this file whole, and the chapter pages light its lines; the first
line is a Colab form title, so the notebooks fold this cell and no chapter's question is answered by its code early). They sit
beside M6's code/training_loop.py, with its names (get_batch, loss_of, train_step, every_window, average_loss), and repeat the
recorded run's own code, training/prompter/train.py: test_training_steps.py checks them against it and against the run's
numbers (training/prompter/unit2.py's export, prompter-unit2.json).
"""
import math
import os
import re

import torch


# ---------------------------------------------------------------------------------------------------- T1 · picking the batches

def check_starts(n_train, n_val, batches=100, B=64, T=32, seed=2024):
    """The checks' fixed windows, as train.py draws them: 100 batches of starts in the training text, then 100 in the held-back
    text."""
    g = torch.Generator().manual_seed(seed)                       # the checks' generator
    train = torch.stack([torch.randint(n_train - T - 1, (B,), generator=g) for _ in range(batches)])   # [batches×B]
    val = torch.stack([torch.randint(n_val - T - 1, (B,), generator=g) for _ in range(batches)])       # [batches×B]
    return train, val


def run_starts(n, steps=100_000, B=64, T=32, seed=1337, make_model=None):
    """Every step's B window starts, as the recorded run drew them (train.py): the main random stream, started by the seed,
    first fills the model's random starting numbers (make_model builds the model), then draws B starts a step."""
    torch.manual_seed(seed)                                       # the main stream: starting numbers, then every batch
    if make_model is not None:
        make_model()                                              # first: the model's random starting numbers
    return torch.stack([torch.randint(n - T - 1, (B,)) for _ in range(steps)])   # [steps×B]  then B starts every step


def times_drawn(starts, n, T=32):
    """How many times each possible start was drawn. randint(n - T - 1) never draws its limit: starts run 0 … n - T - 2."""
    return torch.bincount(starts.flatten(), minlength=n - T - 1)  # [n−T−1]   one count per possible start


def times_asked(starts, n, T=32):
    """How many times each character of the text was a right answer: a window from start i asks about i + 1 … i + T."""
    drawn = times_drawn(starts, n, T)                             # [n−T−1]
    change = torch.zeros(n + 1, dtype=torch.long)
    change[1:n - T] += drawn                                      # a window from i starts asking at i + 1 …
    change[T + 1:n] -= drawn                                      # … and stops after i + T
    return change.cumsum(0)[:n]                                   # [n]       one count per character


def times_through(n, steps=100_000, B=64, T=32):
    """How many times over the run reads its text: every step asks B × T questions, one per right answer."""
    return steps * B * T / n                                      # answers ÷ characters


# ---------------------------------------------------------------------------------------------------- T2 · the loss

def all_at_once(model, x, y):
    """A batch's right answers all at once: their probabilities multiplied, in the model's 32-bit numbers, and their costs,
    minus each one's log, added and averaged (the loss)."""
    probs = torch.softmax(model(x)[0], dim=-1)                    # [B×T×65]  probabilities, as M6's loss_of makes them
    p = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)             # [B×T]     each right answer's probability
    product = p.prod()                                            # []        all of them together: too small for 32 bits, 0.0
    total = -p.log().sum()                                        # []        their costs, added
    return product, total, total / p.numel()                      # the product, the sum, and the average: the loss


def loss_from_scores(model, x, y):
    """The loss as model.py's F.cross_entropy works it out, straight from the scores: for each answer, the log of the total of
    e to every score, minus the right answer's own score. No probability is ever made."""
    scores = model(x)[0]                                          # [B×T×65]  a score for every character
    right = scores.gather(-1, y.unsqueeze(-1)).squeeze(-1)        # [B×T]     each right answer's score
    return (scores.logsumexp(-1) - right).mean()                  # []        minus the log of each probability, averaged


def answer_costs(model, ids, T=32, per=256):
    """Every right answer's probability over a whole text, and whether it was the top pick, read as the final score reads it
    (train.py full_loss): windows side by side, every character scored once but the first and the last few, which fill no
    window."""
    n = len(range(0, len(ids) - T - 1, T))                        # how many whole windows fit
    x, y = ids[:n * T].view(n, T), ids[1:n * T + 1].view(n, T)    # [n×T] each: the inputs, and the targets moved one along
    p, top = [], []
    with torch.no_grad():
        for xb, yb in zip(x.split(per), y.split(per)):
            probs = torch.softmax(model(xb)[0], dim=-1)           # [256×T×65]
            p.append(probs.gather(-1, yb.unsqueeze(-1)).squeeze(-1))   # [256×T]  the right answer's probability
            top.append(probs.argmax(-1) == yb)                    # [256×T]  was it the top pick?
    return torch.cat(p).flatten(), torch.cat(top).flatten()       # [n·T] each


def share_of_loss(p, below=0.01):
    """The answers given a probability under `below`: their share of all the answers, and their share of the loss."""
    cost = -p.log()                                               # each answer's cost
    rare = p < below
    return rare.float().mean().item(), (cost[rare].sum() / cost.sum()).item()


def sure_or_right(p, top):
    """Accuracy, the share of answers whose top pick is right; and the share right but unsure: top pick, yet under one half."""
    return top.float().mean().item(), (top & (p < 0.5)).float().mean().item()


def choices(loss):
    """A loss read backwards: an even guess among n characters costs ln n on every answer, so n = e to the loss."""
    return math.exp(loss)


# ---------------------------------------------------------------------------------------------------- T3 · the way back

def bias_gradient(probs, y):
    """Every readout bias's gradient, by its rule. A bias adds to one character's score at every place, so its gradient is that
    character's probability averaged over the places, minus its share of the right answers."""
    V = probs.size(-1)
    predicted = probs.reshape(-1, V).mean(0)                      # [65]  each character's average probability
    right = torch.bincount(y.flatten(), minlength=V).float() / y.numel()   # [65]  its share of the right answers
    return predicted - right                                      # [65]  average probability, minus share of right answers


def nudge(model, x, y, param, i, h=1e-3):
    """One number's gradient without backpropagation: nudge it up by h and run the batch, then down by h and run it again, and
    take the slope through the two losses. The number is put back as it was."""
    flat = param.data.view(-1)
    old = flat[i].item()
    with torch.no_grad():
        flat[i] = old + h
        up = model(x, y)[1].item()                                # the loss, nudged up
        flat[i] = old - h
        down = model(x, y)[1].item()                              # the loss, nudged down
        flat[i] = old                                             # put back
    return (up - down) / (2 * h)


def grad_twice(model, x, y, param):
    """backward fills every learned number's .grad in one pass, and moves none of them. Run it again without clearing, and the
    gradients add up: why train.py clears them first, with opt.zero_grad()."""
    model.zero_grad()
    model(x, y)[1].backward()                                     # every .grad filled; no number moved
    once = param.grad.clone()
    model(x, y)[1].backward()                                     # again, without clearing
    return once, param.grad.clone()                               # the second: twice the first


def backward_sizes(model, x, y):
    """The gradient on its way back through model.py's pass, part by part: its size (all its numbers measured together) at the
    scores, at the block's top, into each add's branch, below each add, and at the rows the tables made."""
    blk = model.blocks[0]
    rows = model.tok_emb(x) + model.pos_emb(torch.arange(x.size(1)))   # [B×T×28]  token row + position row
    att = blk.attn(blk.ln1(rows))                                 # the attention's branch
    mid = rows + att                                              # its add
    ff = blk.ff(blk.ln2(mid))                                     # the feed-forward's branch
    top = mid + ff                                                # its add: the block's top
    scores = model.ln_f(top) @ model.tok_emb.weight.T + model.head_bias   # [B×T×65]  the readout
    loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
    parts = {"scores": scores, "top": top, "ff_branch": ff, "meet": mid, "attn_branch": att, "rows": rows}
    grads = torch.autograd.grad(loss, list(parts.values()))       # every part's gradient, in one pass back
    return {name: g.norm().item() for name, g in zip(parts, grads)}, dict(zip(parts, grads))


def table_gradient_by_use(model, x, y):
    """The token table's gradient, one job at a time: read in at the start (each input's entry), and reused at the readout (scoring
    every character). Holding one use still (detach) leaves the other's push; the two add up to the whole gradient."""
    W = model.tok_emb.weight                                      # [65×28]

    def push(use):
        rows = (W if use == "in" else W.detach())[x] + model.pos_emb(torch.arange(x.size(1)))
        for b in model.blocks:
            rows = b(rows)
        scores = model.ln_f(rows) @ (W if use == "out" else W.detach()).T + model.head_bias
        loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
        return torch.autograd.grad(loss, W)[0]                    # [65×28]

    return push("in"), push("out")


# ---------------------------------------------------------------------------------------------------- T4 · AdamW

def adamw_step(p, g, m, v, t, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW step for any tensor of numbers, as torch.optim.AdamW takes it on PyTorch's defaults (train.py passes only the
    learning rate). m and v are the running averages of the pushes and of their squares, both 0 before step 1; t counts the
    steps from 1. Returns the moved numbers and the two averages."""
    b1, b2 = betas
    p = p * (1 - lr * weight_decay)                               # the W: weight decay, a pull toward 0 (T4.4)
    m = b1 * m + (1 - b1) * g                                     # the pushes, averaged with their signs (T4.3)
    v = b2 * v + (1 - b2) * g * g                                 # their sizes, squared and averaged
    m_hat = m / (1 - b1 ** t)                                     # both start at 0: the corrections (T4.2)
    v_hat = v / (1 - b2 ** t)
    step = m_hat / (v_hat.sqrt() + eps)                           # each push over its own size: about ±1 at step 1 (T4.1)
    return p - lr * step, m, v


def steps_in_rates(pushes, betas=(0.9, 0.999), eps=1e-8, corrected=True):
    """One number's move at each step, in learning rates (weight decay aside), for a list of its pushes."""
    b1, b2 = betas
    m = v = 0.0
    moves = []
    for t, g in enumerate(pushes, 1):
        m, v = b1 * m + (1 - b1) * g, b2 * v + (1 - b2) * g * g
        if corrected:
            moves.append(-(m / (1 - b1 ** t)) / (math.sqrt(v / (1 - b2 ** t)) + eps))
        else:
            moves.append(-m / (math.sqrt(v) + eps))               # without the corrections
    return moves


def decay_left(rates, weight_decay=0.01):
    """What weight decay alone leaves of a number: each step keeps 1 − weight_decay × that step's learning rate."""
    left = 1.0
    for lr in rates:
        left *= 1 - weight_decay * lr
    return left


def learning_rate(step, lr=3e-3, min_lr=3e-4, warmup=200, max_steps=100_000):
    """train.py's schedule (lr_at): a straight climb to lr over the warm-up, then down a cosine to min_lr at the last step."""
    if step < warmup:
        return lr * step / warmup
    done = min(1.0, (step - warmup) / (max_steps - warmup))
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * done))


def first_steps(model, train_ids, steps, rate=learning_rate, B=64, T=32, watch=None):
    """The run's first steps, as train.py takes them (the seed set just before the model was made): each step's learning rate,
    64 random windows, their loss, the way back, then AdamW's step. Calls watch(model) after every step; returns its answers."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    seen = []
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = rate(step + 1)                          # this step's learning rate
        starts = torch.randint(len(train_ids) - T - 1, (B,))      # this step's 64 places
        x = torch.stack([train_ids[i:i + T] for i in starts])
        y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
        loss = model(x, y)[1]
        opt.zero_grad(set_to_none=True)
        loss.backward()                                           # the way back
        opt.step()                                                # AdamW moves every number
        if watch:
            seen.append(watch(model))
    return seen


@torch.no_grad()
def check_loss(model, ids, starts, T=32):
    """The run's check: the average loss over fixed windows (check_starts), as train.py scores it every 500 steps."""
    model.eval()
    total = sum(model(torch.stack([ids[i:i + T] for i in s]), torch.stack([ids[i + 1:i + T + 1] for i in s]))[1].item() for s in starts)
    model.train()
    return total / len(starts)


# ---------------------------------------------------------------------------------------------------- T5 · the loop

def one_turn(model, opt, train_ids, step, B=64, T=32, clear=True):
    """One turn of train.py's loop, in its order: this step's learning rate, 64 random windows, their loss, the old gradients
    cleared, the way back, then AdamW's step. clear=False leaves the old gradients in (T5.1). Returns the turn's loss."""
    for group in opt.param_groups:
        group["lr"] = learning_rate(step + 1)                     # 1 · this step's learning rate
    starts = torch.randint(len(train_ids) - T - 1, (B,))          # 2 · 64 random places
    x = torch.stack([train_ids[i:i + T] for i in starts])
    y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
    loss = model(x, y)[1]                                         # 3 · their loss
    if clear:
        opt.zero_grad(set_to_none=True)                           # 4 · the last turn's gradients cleared
    loss.backward()                                               # 5 · the way back adds each gradient to what .grad holds
    opt.step()                                                    # 6 · AdamW moves every number
    return loss.item()


def run_turns(model, train_ids, steps, check=None, every=50, clear=True, remake=False):
    """Turns 1 to `steps` from where the model is, scoring check(model) every `every` steps. remake=True makes a new AdamW every
    turn, so its averages and its step count are lost (T5.2). Returns [[step, check], ...]."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    curve = []
    for step in range(steps + 1):
        if check and step % every == 0:
            curve.append([step, check(model)])
        if step == steps:
            return curve
        if remake:
            opt = torch.optim.AdamW(model.parameters(), lr=3e-3)  # a new AdamW: its averages start at 0 again
        one_turn(model, opt, train_ids, step, clear=clear)


def what_lasts(model, opt):
    """What carries on from one turn to the next: the learned numbers, AdamW's two averages for each, and its step count."""
    numbers = sum(p.numel() for p in model.parameters())
    averages = sum(s["exp_avg"].numel() + s["exp_avg_sq"].numel() for s in opt.state.values())
    steps = sorted({int(s["step"]) for s in opt.state.values()})
    return numbers, averages, steps


def saved(path):
    """What a saved file holds: each table's name and shape, how many numbers in all, and the file's size in bytes."""
    tables = torch.load(path, weights_only=True)
    shapes = {name: tuple(t.shape) for name, t in tables.items()}
    return shapes, sum(t.numel() for t in tables.values()), os.path.getsize(path)


# ---------------------------------------------------------------------------------------------------- T6 · the check

def other_checks(model, ids, seeds, batches=100, B=64, T=32):
    """The check on other windows: for each seed, its own 100 batches of 64 starts, drawn as train.py draws the check's, scored
    as check_loss scores them. The model's numbers never change: only the windows do."""
    scores = []
    for seed in seeds:
        g = torch.Generator().manual_seed(seed)                   # another seed: other windows
        starts = [torch.randint(len(ids) - T - 1, (B,), generator=g) for _ in range(batches)]
        scores.append(check_loss(model, ids, starts, T))
    return scores                                                 # one score per seed


def how_often_asked(starts, n, T=32):
    """How many characters of a text the check's windows ask about 0 times, once, twice, …: times_asked over those windows,
    counted by times. The first and the last characters are never asked: no window starts before the first or reaches the
    last."""
    asked = times_asked(starts, n, T)                             # [n]   one count per character
    return torch.bincount(asked).tolist()                         # [how many 0 times, once, twice, …]


def loss_by_position(model, ids, T=32):
    """The final score, position by position: answer_costs reads the whole text in windows of 32, side by side; each
    position's costs are averaged down the windows. Position 0 has one character to go on, position 31 all 32."""
    p, _ = answer_costs(model, ids, T)                            # [n·T]  every right answer's probability, window after window
    return (-p.log()).view(-1, T).mean(0)                         # [T]    one average cost per position


def keep_best(checks, margin=1e-4, patience=20):
    """train.py's rule, replayed over a log of checks ([step, held-back score] pairs): a check that beats the best so far by
    more than `margin` keeps a copy, and the wait starts again; any other check adds 1 to the wait; a wait of `patience`
    would stop the run. Returns the kept steps, the step it would stop at (None: never), and the kept copy's score."""
    best, wait, kept = float("inf"), 0, []
    for step, score in checks:
        if score < best - margin:
            best, wait = score, 0                                 # a new best: keep a copy
            kept.append(step)
        elif step:
            wait += 1                                             # no new best: one more check waited
        if patience and wait >= patience:
            return kept, step, best                               # train.py would stop the run here
    return kept, None, best


# ---------------------------------------------------------------------------------------------------- T7 · the curve

def share_of_fall(checks, step):
    """How much of its whole fall a run's held-back check had made at `step`, read from the check at that step: from the first
    check down to the best one, as a share (0: none yet, 1: all of it). `checks` is a log of [step, held-back score] pairs."""
    first, best = checks[0][1], min(score for _, score in checks)
    now = dict(checks)[step]                                      # the check at that step
    return (first - now) / (first - best)


def worse_than_before(checks):
    """The steps whose check scored worse (higher) than the check just before it, in a log of [step, held-back score] pairs."""
    return [step for (_, before), (step, score) in zip(checks, checks[1:]) if score > before]


def gaps(log):
    """The held-back check minus the training check, at every check: [step, gap] pairs, from a log of [step, training score,
    held-back score]. Both are scored at the same moment, on the same numbers: only the text differs."""
    return [[step, val - train] for step, train, val in log]


def write(model, tok, n=1000, seed=42, temperature=1.0):
    """What a model writes from a new line: n characters, picked with its own generator, as the course's writing samples are
    made (training/prompter/unit2.py). The same start, seed and temperature every time, so only the numbers differ."""
    model.eval()
    with torch.no_grad():
        out = model.generate(torch.tensor([tok.encode("\n")]), n, temperature, torch.Generator().manual_seed(seed))
    return tok.decode(out[0, 1:].tolist())                        # the new line it started from left out


def word_share(text, training_text):
    """The share of a text's words found in the training text, and the words found nowhere in it. A word is a run of letters
    and apostrophes, read in lower case. Found means spelt somewhere in the training text: it says nothing about sense."""
    known = set(re.findall(r"[a-z']+", training_text.lower()))   # every word the training text spells
    words = re.findall(r"[a-z']+", text.lower())
    found = sum(w in known for w in words)
    return found / len(words), sorted({w for w in words if w not in known})

In [6]:
tok = Tokenizer(text)
cfg = Config(vocab_size=len(tok.chars))
ids = torch.tensor(tok.encode(text))                        # the whole text as ids
cut = int(0.9 * len(ids))
train_ids, val_ids = ids[:cut], ids[cut:]                   # the first 90% trains; the rest is held back
training_text = tok.decode(train_ids.tolist())              # the training text, as characters again

def find(*names):
    """A file the run left: in the labs' model folder, or in the course's results."""
    paths = [os.path.join(here, '..', 'model', names[0]),
             os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'results', 'prompter-r1', names[1])]
    found = [p for p in paths if os.path.exists(p)]
    assert found, f"{names[0]} is not here: run the Setup cell at the top"
    return found[0]
log = [[e['step'], e['train'], e['val']] for e in json.load(open(find('prompter-r1-train.json', 'train.json')))['log']]
checks = [[step, val] for step, _, val in log]              # the held-back checks alone: [step, score]
print(f"{len(log)} checks in the log, steps {log[0][0]} to {log[-1][0]:,} · the first: {log[0]}")
assert len(log) == 201

201 checks in the log, steps 0 to 100,000 · the first: [0, 4.17859, 4.17804]


# Section 1 · How the curve falls

How fast the run's held-back check fell, and how often a check was worse than the one before. Each chapter below follows its page:
Picture it, Watch it in our run, then Read and run the code.

## T7.1 · How fast it falls

*How fast does the loss fall?*

**Predict first** (the page asks it too, before its clip): the run's held-back check fell from 4.18 at step 0 to its best, 1.935, at step 98,000. By step 500,
just 1 in 200 of the run's steps, how much of that fall was done: most of it, more than 4 in 5; about half of it, the rest
came later; or about 1 in 200 of it, steady, step by step?

### T7.1.1 · Picture it

A training curve draws the loss, step by step. At any step, how much of its fall is done? Picture a made-up loss that falls
from 10 to 2: a fall of 8. At one step, it reads 4: it has fallen 6 of the 8. So 6 out of 8, or 3 out of 4, of its fall is done.
(The page's loss is made up to show the idea.)

### T7.1.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the run's 201 held-back checks. They fall from 4.18 at step 0 to their best, 1.935, at
step 98,000. By step 500, the check was 2.30: 84 in 100 of the whole fall was done. By step
5,000, 2.02: 96 in 100. All the steps after made only the last small part. Counting pairs
knows only which character follows which: 2.48. By step 500, the check was already below it.

### T7.1.3 · Read and run the code

`share_of_fall` in `code/training_steps.py` (copied in above) reads how much of a fall is done. It takes the first check, the best
check, and the check at the step you pick. It divides the fall so far by the whole fall. On the run's log, 0.836 at step 500,
and 0.963 at step 5,000. It needs no model, only the log: the run wrote every check down as it went.

In [7]:
for step in (0, 500, 1000, 2000, 5000, 20000, 98000):
    share = share_of_fall(checks, step)
    print(f"step {step:>6,}: {dict(checks)[step]:.4f} · {share:.3f} of the fall  " + "#" * round(40 * share))
assert abs(share_of_fall(checks, 500) - 0.835709) < 1e-5 and abs(share_of_fall(checks, 5000) - 0.963492) < 1e-5
print(f"counting pairs: 2.4819 · the check at step 500: {dict(checks)[500]:.4f}")

step      0: 4.1780 · 0.000 of the fall  
step    500: 2.3036 · 0.836 of the fall  #################################
step  1,000: 2.1243 · 0.916 of the fall  #####################################
step  2,000: 2.0532 · 0.947 of the fall  ######################################
step  5,000: 2.0170 · 0.963 of the fall  #######################################
step 20,000: 1.9912 · 0.975 of the fall  #######################################
step 98,000: 1.9351 · 1.000 of the fall  ########################################
counting pairs: 2.4819 · the check at step 500: 2.3036


### ✋ Your turn

How much of the fall was done by step 1,000? Put it in `answer`.

In [8]:
answer = None   # TODO: share_of_fall(checks, 1000)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - share_of_fall(checks, 1000)) < 1e-12, 'the share at that step: share_of_fall(checks, 1000)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [9]:
#@title Show a solution { display-mode: "form" }
answer = share_of_fall(checks, 1000)
print(f'by step 1,000: {answer:.3f} of the fall')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - share_of_fall(checks, 1000)) < 1e-12, 'the share at that step: share_of_fall(checks, 1000)'
    print("✓ right:", answer)

by step 1,000: 0.916 of the fall
✓ right: 0.9156512628458056


## T7.2 · Ups and downs

*Does every check beat the one before?*

**Predict first** (the page asks it too, before its clip): after step 0, the run checked 200 more times. How many of those checks scored worse than the check just before: about
90, nearly half; about 20, a few, mostly early; or about 5, hardly any?

### T7.2.1 · Picture it

Does every check beat the one before it? Picture 6 made-up checks: 3.0, 2.6, 2.4, 2.5, 2.3, 2.2. One of them, 2.5, is worse than
the check before. Yet the line still falls, from 3.0 to 2.2. One worse check is not a run going wrong. (The page's
checks are made up to show the idea.)

### T7.2.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the run's held-back checks, zoomed in. Each one worse than the check before is lit. Of the 200 checks after
step 0, 88 were worse than the one before. Just 1 of the first 10, to step 5,000. Then
87 of the next 190: nearly half. Each step moves the numbers to suit its own 64 random windows,
which can make other text score a little worse. After step 5,000, that wobble outweighs the small gain from one check to the next.

### T7.2.3 · Read and run the code

`worse_than_before` finds every check worse than the one before. It walks the log in order, and compares each check with the one
just before it. On the run's log, it finds 88. The first is at step 3,000. One check against the last says little: read
the curve over many checks.

In [10]:
worse = worse_than_before(checks)
print(f"{len(worse)} of {len(checks) - 1} checks worse than the one before · the first: step {worse[0]:,}")
print(f"to step 5,000: {sum(s <= 5000 for s in worse)} of 10 · after: {sum(s > 5000 for s in worse)} of {len(checks) - 11}")
for lo in range(0, 100000, 20000):
    n = sum(lo < s <= lo + 20000 for s in worse)
    print(f"steps {lo + 500:>6,} to {lo + 20000:>7,}: {n:>2} of 40  " + "#" * n)
assert len(worse) == 88 and worse[0] == 3000
assert sum(s <= 5000 for s in worse) == 1 and sum(s > 5000 for s in worse) == 87

88 of 200 checks worse than the one before · the first: step 3,000
to step 5,000: 1 of 10 · after: 87 of 190
steps    500 to  20,000: 14 of 40  ##############
steps 20,500 to  40,000: 18 of 40  ##################
steps 40,500 to  60,000: 16 of 40  ################
steps 60,500 to  80,000: 18 of 40  ##################
steps 80,500 to 100,000: 22 of 40  ######################


### ✋ Your turn

Of the 100 checks after step 50,000, how many were worse than the one before? Put the number in `answer`.

In [11]:
answer = None   # TODO: sum(s > 50000 for s in worse)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == sum(s > 50000 for s in worse), 'count the worse steps past 50,000: sum(s > 50000 for s in worse)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [12]:
#@title Show a solution { display-mode: "form" }
answer = sum(s > 50000 for s in worse)
print(f'after step 50,000: {answer} of 100 worse than the one before')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == sum(s > 50000 for s in worse), 'count the worse steps past 50,000: sum(s > 50000 for s in worse)'
    print("✓ right:", answer)

after step 50,000: 48 of 100 worse than the one before
✓ right: 48


# Section 2 · Beyond one curve

The held-back curve beside the training text's curve, and beside what the model wrote.

## T7.3 · The gap between two curves

*What does the gap between the two curves say?*

**Predict first** (the page asks it too, before its clip): every check scores both texts. At step 500, the training check and the held-back check were just 0.005 apart. What does
that gap do over the rest of the run: it grows for a while, then stays about the same; it keeps growing just as fast, all the way to
the end; or it stays about as small, as the two texts are alike?

### T7.3.1 · Picture it

Every check scores two texts: the one the model trains on, and the held-back one. What does the space between them say? Picture a
made-up learner, scored on questions it practises, and on new ones. Its loss on the practised questions is 2; on new ones, 3. The
gap, 1: it does better on questions it has practised than on new ones. (The page's learner is made up to show the idea.)

### T7.3.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are both checks of the run: the training text's, and the held-back text's, from step 500 on. At step 500, they were
0.005 apart. By step 5,000, 0.13. From step 20,000 on, the gap stayed between 0.16
and 0.18: it barely grew. It scores better on the text it trains on, yet the held-back check kept falling, to
its best at step 98,000.

### T7.3.3 · Read and run the code

`gaps` takes the held-back check minus the training check. Both are scored at the same moment, on the same numbers: only the text
differs. On the run's log: 0.005 at step 500, and 0.13 at step 5,000. The largest gap,
0.18, came at step 63,000. At the end, `train.py` scored best.pt on both whole texts: training text
1.765, held-back 1.932: 0.17 apart. (Some of the gap may be the texts themselves: the
held-back text has characters' names the training text never has.)

In [13]:
g = dict((step, gap) for step, gap in gaps(log))
for step in (500, 1000, 2000, 5000, 10000, 20000, 50000, 98000, 100000):
    train_, val_ = dict((s, (t, v)) for s, t, v in log)[step]
    print(f"step {step:>7,}: training {train_:.4f} · held-back {val_:.4f} · gap {g[step]:.4f}  " + "#" * round(200 * g[step]))
late = [gap for step, gap in g.items() if step >= 20000]
print(f"from step 20,000: {min(late):.3f} to {max(late):.3f}")
assert abs(g[500] - 0.00532) < 1e-5 and abs(g[5000] - 0.13219) < 1e-5
assert abs(min(late) - 0.15688) < 1e-5 and abs(max(late) - 0.17999) < 1e-5

step     500: training 2.2983 · held-back 2.3036 · gap 0.0053  #
step   1,000: training 2.0861 · held-back 2.1243 · gap 0.0382  ########
step   2,000: training 1.9709 · held-back 2.0532 · gap 0.0822  ################
step   5,000: training 1.8848 · held-back 2.0170 · gap 0.1322  ##########################
step  10,000: training 1.8531 · held-back 2.0106 · gap 0.1574  ###############################
step  20,000: training 1.8225 · held-back 1.9912 · gap 0.1687  ##################################
step  50,000: training 1.7934 · held-back 1.9654 · gap 0.1721  ##################################
step  98,000: training 1.7649 · held-back 1.9351 · gap 0.1702  ##################################
step 100,000: training 1.7654 · held-back 1.9360 · gap 0.1707  ##################################
from step 20,000: 0.157 to 0.180


### ✋ Your turn

At which step was the gap largest? Put the step in `answer`.

In [14]:
answer = None   # TODO: max(g, key=g.get)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 63000, 'the step whose gap is largest: max(g, key=g.get)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [15]:
#@title Show a solution { display-mode: "form" }
answer = max(g, key=g.get)
print(f'the largest gap: {g[answer]:.4f}, at step {answer:,}')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 63000, 'the step whose gap is largest: max(g, key=g.get)'
    print("✓ right:", answer)

the largest gap: 0.1800, at step 63,000
✓ right: 63000


## T7.4 · The writing

*How does the writing change as the run goes on?*

**Predict first** (the page asks it too, before its clip): from step 5,000 to its best at step 98,000, the held-back check fell only 0.08 more. What happened to the writing over those
steps: more of its words were found in the training text; about the same share, as the loss hardly moved; or a smaller share, as
it tried longer, rarer words?

### T7.4.1 · Picture it

The check gives a number. How can the writing itself be judged? Picture a made-up line of 6 words: the kng sat on hs bicycle. Look
each word up in the training text: the, sat and on are there. The other 3 are not. So 3 of 6, half, are found. Bicycle is spelt
right, yet not there. Found says nothing about sense. (The page's line is made up to show the idea.)

### T7.4.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is what the model wrote, from a new line, at steps 0, 500, 5,000 and 98,000. At step 0, 5 in 100 of its words are
found in the training text. At step 500, 19. At step 5,000, 55 in 100. At step 98,000, best.pt:
69 in 100. The loss averages one character at a time. A word is found only if all its letters fit, so small gains on each letter multiply.
(The
writing at steps 0 to 5,000 was replayed: the run re-trained from its seed and matched to its log. This notebook has best.pt only.)

### T7.4.3 · Read and run the code

`write` starts from a new line, with seed 42 and temperature 1, and writes 1,000 characters. `word_share` looks each word up in the
training text. On best.pt's writing: 0.686. Fixed settings, like the check's fixed windows, let two samples
differ only because the numbers did. (About ten seconds.)

In [16]:
best = Prompter(cfg)
best.load_state_dict(torch.load(find('prompter-r1-best.pt', 'best.pt'), weights_only=True))   # the numbers train.py kept
wrote = write(best, tok)
share, nowhere = word_share(wrote, training_text)
print(wrote[:400])
print("...")
print(f"{share:.3f} of its words found in the training text · found nowhere: {', '.join(nowhere[:12])}, ...")
assert wrote == "\nAs loud save may: whom grows no, onoth, but court of in and go lamstre!\n\nKISS:\nWhat all thegnings?\n\nDUKE VINCENTIO:\nAway\nKING HE:\nIt hand the how by the your for not prenatnge, likes both.\n\nPOMEOK:\nThat?\n\nGLOUCETES:\nThe of his he like ward\nThey home diff the agamen\nseequeent would, by thous ance breast thour fall 'names even panre to wife,\nAnd tistly that bite's he\nMusided,-\nSow, Frown. did year miscauth and the him,\nAnd that nightt Heim, wid, is not now, botheifixendunward?\n\nPry sure were what tame lid's accte witral maintep some\nRivetalk'd; undes, gramsmannius?\n\nDUKE VINCENTIO:\nAs me it kit good. And resey'ks whis toge.\n\nShalt, hall thou light neant their sweeanotly shicers.\n\nKINGSIR. Here boudger her, maked which cleace pranned I hissetives:\nI thther mell when, I count love commenelianture dluse givalles?\nAnd so would do, mas-had the for thy withou, and I\nexpes:\nRome, that, have there you art theses be the your men:'\nOut of her firse dawlinged hath, servey courtion,\nI but my trean "
assert abs(share - 0.6857) < 1e-4


As loud save may: whom grows no, onoth, but court of in and go lamstre!

KISS:
What all thegnings?

DUKE VINCENTIO:
Away
KING HE:
It hand the how by the your for not prenatnge, likes both.

POMEOK:
That?

GLOUCETES:
The of his he like ward
They home diff the agamen
seequeent would, by thous ance breast thour fall 'names even panre to wife,
And tistly that bite's he
Musided,-
Sow, Frown. did year 
...
0.686 of its words found in the training text · found nowhere: 'names, accte, agamen, ance, bite's, botheifixendunward, boudger, cleace, commenelianture, courtion, dawlinged, diff, ...


### ✋ Your turn

Write a line of your own: some words misspelt, some spelt right but rare. Count its words found in the training text, and put
the share in `answer`.

In [17]:
mine = 'the kng sat on hs bicycle'   # change it
answer = None   # TODO: word_share(mine, training_text)[0]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - word_share(mine, training_text)[0]) < 1e-12, 'the share, the first of the two things word_share gives: word_share(mine, training_text)[0]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [18]:
#@title Show a solution { display-mode: "form" }
mine = 'the kng sat on hs bicycle'
answer, missing = word_share(mine, training_text)
print(f'{answer:.2f} found · found nowhere: {missing}')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - word_share(mine, training_text)[0]) < 1e-12, 'the share, the first of the two things word_share gives: word_share(mine, training_text)[0]'
    print("✓ right:", answer)

0.50 found · found nowhere: ['bicycle', 'hs', 'kng']
✓ right: 0.5


The run's held-back curve fell fast, then slowly: 84 in 100 of its fall by step 500, 96 in 100 by step 5,000. After step 5,000 it wobbled, 87 of 190 checks worse than the
one before, while still falling over many checks. Beside the training text's curve, the gap grew, then nearly stopped growing,
between 0.16 and 0.18. And while the loss barely fell after step 5,000, more of the writing's
words were found in the training text: 55 to 69 in 100.

Next: **T8 · Train your own tiny transformer**, the same run, yours.